In [2]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
pc_features = np.load('/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/pc_results/pca_results.npz', allow_pickle=True)
image_inf = pd.read_csv("/media/ubuntu/sda/Monkey/data/train_image_MonkeyF.csv")

In [4]:
features_40d = pc_features['features_40d']

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 提取前10个PCA维度
features_10d = features_40d[:, :10]
print(f"特征维度: {features_10d.shape}")

# 进行K-means聚类，分成10个cluster
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_10d)



特征维度: (22248, 10)


In [6]:
# 根据规则定义class到cluster的映射：如果某个class有超过10张图片在某个cluster中，认为这个class属于这个cluster
threshold = 10  # 阈值：超过10张图片
classes = image_inf['class'].unique()

# 统计每个class在每个cluster中的数量
class_cluster_matrix = {}
class_to_cluster = {}  # 记录每个class属于哪个cluster

for class_name in classes:
    class_indices = image_inf[image_inf['class'] == class_name].index.values
    class_cluster_counts = {}
    
    # 统计该class在每个cluster中的数量
    for cluster_id in range(n_clusters):
        cluster_mask = cluster_labels == cluster_id
        cluster_indices = np.where(cluster_mask)[0]
        
        # 计算该class在这个cluster中的数量
        count = len(np.intersect1d(class_indices, cluster_indices))
        class_cluster_counts[cluster_id] = count
    
    class_cluster_matrix[class_name] = class_cluster_counts
    
    # 找到超过阈值的cluster
    matching_clusters = [cid for cid, count in class_cluster_counts.items() if count > threshold]
    
    if len(matching_clusters) == 1:
        # 只有一个cluster超过阈值，直接分配
        class_to_cluster[class_name] = matching_clusters[0]
    elif len(matching_clusters) > 1:
        # 多个cluster超过阈值，选择数量最多的
        best_cluster = max(matching_clusters, key=lambda x: class_cluster_counts[x])
        class_to_cluster[class_name] = best_cluster
    else:
        # 没有cluster超过阈值，选择数量最多的cluster
        best_cluster = max(class_cluster_counts.items(), key=lambda x: x[1])[0]
        class_to_cluster[class_name] = best_cluster

print(f"总共 {len(classes)} 个类别")
print(f"阈值设置: 超过 {threshold} 张图片")
print(f"\n类别到Cluster的映射统计:")

# 统计每个cluster包含多少个class
cluster_class_counts = {}
for class_name, assigned_cluster in class_to_cluster.items():
    if assigned_cluster not in cluster_class_counts:
        cluster_class_counts[assigned_cluster] = []
    cluster_class_counts[assigned_cluster].append(class_name)

for cluster_id in range(n_clusters):
    count = len(cluster_class_counts.get(cluster_id, []))
    print(f"  Cluster {cluster_id}: {count} 个类别")


总共 1854 个类别
阈值设置: 超过 10 张图片

类别到Cluster的映射统计:
  Cluster 0: 148 个类别
  Cluster 1: 19 个类别
  Cluster 2: 206 个类别
  Cluster 3: 165 个类别
  Cluster 4: 116 个类别
  Cluster 5: 36 个类别
  Cluster 6: 104 个类别
  Cluster 7: 141 个类别
  Cluster 8: 915 个类别
  Cluster 9: 4 个类别


In [7]:
# 保存class到cluster的映射结果
# 创建映射DataFrame
mapping_df = pd.DataFrame({
    'class': list(class_to_cluster.keys()),
    'assigned_cluster': list(class_to_cluster.values())
})

# 添加每个class在分配到的cluster中的图片数量
image_counts = []
for class_name in mapping_df['class']:
    class_indices = image_inf[image_inf['class'] == class_name].index.values
    assigned_cluster = class_to_cluster[class_name]
    cluster_mask = cluster_labels == assigned_cluster
    cluster_indices = np.where(cluster_mask)[0]
    count = len(np.intersect1d(class_indices, cluster_indices))
    image_counts.append(count)

mapping_df['images_in_cluster'] = image_counts
mapping_df['meets_threshold'] = mapping_df['images_in_cluster'] > threshold

# 按cluster排序
mapping_df = mapping_df.sort_values('assigned_cluster')

# 保存为CSV
mapping_output_path = '/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/class_to_cluster_mapping.csv'
mapping_df.to_csv(mapping_output_path, index=False)
print(f"类别到Cluster的映射已保存到: {mapping_output_path}")

# 保存为字典格式（npz）
mapping_dict = {
    'class_to_cluster': class_to_cluster,
    'cluster_to_classes': cluster_class_counts,
    'threshold': threshold,
    'class_cluster_matrix': class_cluster_matrix
}

mapping_npz_path = '/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/class_to_cluster_mapping.npz'
np.savez(mapping_npz_path, **mapping_dict)
print(f"映射字典已保存到: {mapping_npz_path}")


类别到Cluster的映射已保存到: /media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/class_to_cluster_mapping.csv
映射字典已保存到: /media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/class_to_cluster_mapping.npz


In [8]:
# 使用UMAP进行降维可视化
try:
    import umap
    print("UMAP库已导入")
except ImportError:
    print("正在安装UMAP库...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'umap-learn'])
    import umap
    print("UMAP库安装完成")

# 使用前10个PCA维度进行UMAP降维（与K-means聚类使用的特征一致）
print(f"输入特征维度: {features_10d.shape}")

# 创建UMAP模型
reducer = umap.UMAP(n_components=3, 
                    random_state=3,
                    n_neighbors=15,
                    min_dist=0.1,
                    metric='euclidean')

# 执行降维
print("正在进行UMAP降维...")
umap_embedding = reducer.fit_transform(features_10d)
print(f"UMAP降维完成，输出维度: {umap_embedding.shape}")


UMAP库已导入
输入特征维度: (22248, 10)
正在进行UMAP降维...


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP降维完成，输出维度: (22248, 3)


In [9]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt

# 生成tab20颜色
n_clusters = len(np.unique(cluster_labels))
tab20_colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
# 将matplotlib RGBA转换为plotly可用的颜色格式
tab20_hex = [f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{a})' 
             for r, g, b, a in tab20_colors]

# 创建颜色映射字典
color_map = {i: tab20_hex[i] for i in range(n_clusters)}

# 创建数据
data = []
for i in range(n_clusters):
    cluster_mask = cluster_labels == i
    trace = go.Scatter3d(
        x=umap_embedding[cluster_mask, 0],
        y=umap_embedding[cluster_mask, 1],
        z=umap_embedding[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=tab20_hex[i],  # 使用tab20颜色
            opacity=1,
            line=dict(width=0)
        ),
        name=f'Cluster {i}',
        hovertemplate=(
            'Cluster: %{text}<br>' +
            'UMAP1: %{x:.3f}<br>' +
            'UMAP2: %{y:.3f}<br>' +
            'UMAP3: %{z:.3f}<extra></extra>'
        ),
        text=[f'Cluster {i}'] * np.sum(cluster_mask)
    )
    data.append(trace)

layout = go.Layout(
    title=dict(
        text='3D UMAP Visualization Colored by K-means Clusters',
        font=dict(size=20, family='Arial', color='black'),
        x=0.5
    ),
    scene=dict(
        xaxis=dict(
            title='UMAP 1',
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white'
        ),
        yaxis=dict(
            title='UMAP 2',
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white'
        ),
        zaxis=dict(
            title='UMAP 3',
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white'
        ),
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        ),
        aspectmode='data'
    ),
    legend=dict(
        title=dict(text='Clusters'),
        font=dict(size=12),
        itemsizing='constant',
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top'
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=200, t=50, b=0),
    hovermode='closest'
)

# 创建图形
fig = go.Figure(data=data, layout=layout)

# 显示图形
fig.show()


In [10]:
# 合并cluster：cluster 1和6合并，cluster 7和4合并
print("原始cluster数量:", n_clusters)
print("合并规则:")
print("  - Cluster 1 和 Cluster 6 合并")
print("  - Cluster 7 和 Cluster 4 合并")

# 创建合并后的cluster标签
merged_cluster_labels = cluster_labels.copy()

# 合并策略：将较小的cluster编号合并到较大的cluster编号
# Cluster 1 和 6 -> 合并到 1
merged_cluster_labels[merged_cluster_labels == 7] = 1

# Cluster 7 和 4 -> 合并到 4
merged_cluster_labels[merged_cluster_labels == 9] = 6
merged_cluster_labels[merged_cluster_labels == 5] = 4

# 重新映射cluster编号，使其连续（0-7）
# 原始cluster: 0, 1(包含6), 2, 3, 4(包含7), 5, 8, 9
# 需要重新映射为: 0, 1, 2, 3, 4, 5, 6, 7

# 创建映射字典
cluster_mapping = {
    0: 0,
    1: 1,  # 包含原来的1和6
    2: 2,
    3: 3,
    4: 4,  # 包含原来的4和7
    6: 5,
    8: 6
}

# 应用映射
merged_cluster_labels_renumbered = np.array([cluster_mapping[label] for label in merged_cluster_labels])

# 更新cluster数量
n_merged_clusters = len(set(merged_cluster_labels_renumbered))
print(f"\n合并后cluster数量: {n_merged_clusters}")



原始cluster数量: 10
合并规则:
  - Cluster 1 和 Cluster 6 合并
  - Cluster 7 和 Cluster 4 合并

合并后cluster数量: 7


In [11]:
# 更新class到cluster的映射（基于合并后的cluster）
# 重新计算每个class在合并后cluster中的分布
merged_class_to_cluster = {}
merged_cluster_class_counts = {i: [] for i in range(n_merged_clusters)}

for class_name in classes:
    class_indices = image_inf[image_inf['class'] == class_name].index.values
    class_cluster_counts = {}
    
    # 统计该class在每个合并后cluster中的数量
    for cluster_id in range(n_merged_clusters):
        cluster_mask = merged_cluster_labels_renumbered == cluster_id
        cluster_indices = np.where(cluster_mask)[0]
        count = len(np.intersect1d(class_indices, cluster_indices))
        class_cluster_counts[cluster_id] = count
    
    # 找到超过阈值的cluster
    matching_clusters = [cid for cid, count in class_cluster_counts.items() if count > threshold]
    
    if len(matching_clusters) == 1:
        merged_class_to_cluster[class_name] = matching_clusters[0]
    elif len(matching_clusters) > 1:
        best_cluster = max(matching_clusters, key=lambda x: class_cluster_counts[x])
        merged_class_to_cluster[class_name] = best_cluster
    else:
        best_cluster = max(class_cluster_counts.items(), key=lambda x: x[1])[0]
        merged_class_to_cluster[class_name] = best_cluster
    
    # 添加到cluster的类别列表
    assigned_cluster = merged_class_to_cluster[class_name]
    merged_cluster_class_counts[assigned_cluster].append(class_name)



In [22]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt

# 生成tab20颜色
n_clusters = len(np.unique(merged_cluster_labels_renumbered))
tab20_colors = ["#e19368",
"#adba7d",
"#db98d5",
"#d36b6b",
"#608eb7",
"#9cbd13",
'#e3d696']
# 将matplotlib RGBA转换为plotly可用的颜色格式

# 创建颜色映射字典
color_map = {i: tab20_colors[i] for i in range(n_clusters)}

# 创建数据
data = []
for i in range(n_clusters):
    cluster_mask = merged_cluster_labels_renumbered == i
    trace = go.Scatter3d(
        x=umap_embedding[cluster_mask, 0],
        y=umap_embedding[cluster_mask, 1],
        z=umap_embedding[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=tab20_colors[i],  # 使用tab20颜色
            opacity=1,
            line=dict(width=0)
        ),
        name=f'Cluster {i}',
        hovertemplate=(
            'Cluster: %{text}<br>' +
            'UMAP1: %{x:.3f}<br>' +
            'UMAP2: %{y:.3f}<br>' +
            'UMAP3: %{z:.3f}<extra></extra>'
        ),
        text=[f'Cluster {i}'] * np.sum(cluster_mask)
    )
    data.append(trace)

layout = go.Layout(
    title=dict(
        text='3D UMAP Visualization Colored by K-means Clusters',
        x=0.5
    ),
    scene=dict(
        xaxis=dict(
            title='UMAP 1',
            showgrid=False,  # 去除网格线
            zeroline=False,  # 去除零线
            showbackground=False,  # 去除背景
            showticklabels=True
        ),
        yaxis=dict(
            title='UMAP 2',
            showgrid=False,  # 去除网格线
            zeroline=False,  # 去除零线
            showbackground=False,  # 去除背景
            showticklabels=True,
        ),
        zaxis=dict(
            title='UMAP 3',
            showgrid=False,  # 去除网格线
            zeroline=False,  # 去除零线
            showbackground=False,  # 去除背景
            showticklabels=True
        ),
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        ),
        aspectmode='data',
        bgcolor='white'  # 设置背景为白色
    ),
    legend=dict(
        title=dict(text='Clusters'),
        font=dict(size=12),
        itemsizing='constant',
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top'
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=200, t=50, b=0),
    hovermode='closest',
    plot_bgcolor='white',  # 设置绘图区域背景为白色
    paper_bgcolor='white'  # 设置纸张背景为白色
)

# 创建图形
fig = go.Figure(data=data, layout=layout)

# 显示图形
fig.show()

# 保存为PDF文件


In [51]:
merged_cluster_class_counts

{0: ['acorn',
  'almond',
  'aloe',
  'apple',
  'apple_tree',
  'artichoke',
  'arugula',
  'asparagus',
  'avocado',
  'banana',
  'basil',
  'bead',
  'bean',
  'beet',
  'bell_pepper',
  'berry',
  'blackberry',
  'blueberry',
  'bok_choy',
  'broccoli',
  'brussels_sprouts',
  'cabbage',
  'cactus',
  'candy',
  'candy_cane',
  'cantaloupe',
  'carrot',
  'cashew',
  'cauliflower',
  'celery',
  'cherry',
  'chive',
  'cilantro',
  'cinnamon',
  'clam',
  'clove',
  'clover',
  'coconut',
  'cocoon',
  'coffee_bean',
  'compost',
  'corn',
  'cornhusk',
  'cornucopia',
  'cranberry',
  'cucumber',
  'daisy',
  'dandelion',
  'easter_egg',
  'egg',
  'eggplant',
  'eggshell',
  'fern',
  'fig',
  'flower',
  'fruit',
  'fungus',
  'garlic',
  'gem',
  'ginger',
  'gold',
  'gourd',
  'grain',
  'grape',
  'grapefruit',
  'grass',
  'gravel',
  'green_beans',
  'gumball',
  'gumdrop',
  'honeycomb',
  'jalapeno',
  'jelly_bean',
  'kale',
  'kiwi',
  'leaf',
  'leek',
  'lemon',
  '

In [23]:
fig.write_image("/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyF/3d_umap_clusters.pdf", format='pdf',width=1200,
    height=900,
    scale=3)